Q1. 
(a) None
(b) '2026-05-06'
(c) ['2026-05-06', '2026-05-18']
(d) [('2026', '05', '06'), ('2026', '05', '18')]
(e) ['2026-05-06', '2026-05-18']

(c), (d), (e)가 같은 입력에서 서로 다른 형태의 결과를 돌려주는 이유는, re.findall은 캡처 그룹 (...)이 있으면 그룹 단위로 추출해 튜플 리스트를 반환하고, 없으면 전체 매칭 문자열의 리스트를 반환한다. 비캡처 그룹 (?:...)은 캡처 그룹으로 취급되지 않으므로 (e)는 그룹이 없는 (c)와 동일한 형태의 결과를 낸다.

Q2. 
(a) '[T]!'
(b)'[T]안녕[T] [T]세상[T]!'
(c) '[T]안녕[T] [T]세상[T]!'
(d) '수강생 <30>명, 조교 <3>명'
(e) '수강생 <\x01>명, 조교 <\x01>명'

(i) (a)와 (b)의 차이
.+는 탐욕적 수량자라 첫 <부터 마지막 >까지 최대한 길게 매칭하는 반면, .+?는 게으른 수량자라 <에서 시작해 가장 가까운 >에서 멈추므로 태그 하나하나를 개별적으로 치환한다.
(ii) (d)와 (e)의 차이
r"<\1>"은 원시 문자열이라 \1이 regex 역참조로 해석되지만, "<\1>"은 일반 문자열이라 파이썬이 \1을 먼저 ASCII 제어문자(\x01)로 변환해버리므로 역참조가 동작하지 않는다.

In [4]:
# Q3
import re
from collections import Counter

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ", 
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. ",
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    "  여러  공백과\n\n\n줄바꿈이  많은  텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

#(a) clean_post(post: str) -> str
_URL     = re.compile(r'https?://\S+')
_HTML    = re.compile(r'<[^>]+>')
_EMAIL   = re.compile(r'[\w.+-]+@[\w.-]+\.\w+')
_PHONE   = re.compile(r'\d{2,4}-\d{3,4}-\d{4}')
_MENTION = re.compile(r'@\w+')
_HASH    = re.compile(r'#\w+')
_JAMO    = re.compile(r'[\u3131-\u3163]+')
_SPACE   = re.compile(r'\s+')


def clean_post(post: str) -> str:
    post = _URL.sub(' ', post)           # 1. URL → 공백
    post = _HTML.sub('', post)           # 2. HTML 태그 제거
    post = _EMAIL.sub('[이메일]', post)   # 3a. 이메일 마스킹
    post = _PHONE.sub('[전화]', post)     # 3b. 전화번호 마스킹
    post = _MENTION.sub(' ', post)       # 4a. 멘션 → 공백
    post = _HASH.sub(' ', post)          # 4b. 해시태그 → 공백
    post = _JAMO.sub('', post)           # 5. 자음/모음 제거
    post = _SPACE.sub(' ', post).strip() # 6. 공백 정리
    print(post)
    
    return post

# (b) extract_hashtags(post: str) -> list[str]

def extract_hashtags(post: str) -> list[str]:
    return [tag[1:] for tag in _HASH.findall(post)]

# (c)  analyze_posts(posts: list[str]) -> dict

def analyze_posts(posts: list[str]) -> dict:
    cleaned = [clean_post(p) for p in posts]
    avg_length = round(sum(len(c) for c in cleaned) / len(cleaned), 2)

    all_tags = []
    for p in posts:
        all_tags.extend(extract_hashtags(p))
    hashtag_counts = dict(Counter(all_tags).most_common())

    masked_count = 0
    for p in posts:
        _, n1 = _EMAIL.subn('[이메일]', p)
        _, n2 = _PHONE.subn('[전화]', p)
        masked_count += n1 + n2

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_length,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count,
    }


print(analyze_posts(posts))

오늘 수업 진짜 재밌었음!! 감사
자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강.
문의는 [이메일] ([전화])로!
여러 공백과 줄바꿈이 많은 텍스트
진짜 좋다
{'posts_n': 7, 'avg_length_after_clean': 13.57, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


[1] '오늘 수업 진짜 재밌었음!! 감사 자료:'
[2] '팀플 어디서 모이지 카톡'
[3] '중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!'
[4] '여러 공백과 줄바꿈이 많은 텍스트'
[5] '진짜 좋다'

{
    "posts_n": 7,
    "avg_length_after_clean": 13.57,
    "hashtag_counts": {"파이썬": 2, "DCCP2026": 1, "팀플": 1, "추천": 1},
    "masked_count": 2
}

3단계(이메일 마스킹)와 4단계(멘션 제거)의 순서를 바꾸면 문제가 생겨. mam3b@snu.ac.kr에서 @\w+(멘션 패턴)가 @snu를 먼저 잡아 공백으로 치환해버리면, 이메일 패턴이 mam3b snu.ac.kr처럼 깨진 문자열을 보게 되어 이메일을 제대로 마스킹하지 못하게 된다. 즉, 단계 4를 단계 3보다 먼저 실행하면 개인정보 마스킹이 누락될 수 있다.

참조한 생성형 ai: https://claude.ai/share/6de30095-2259-4735-958b-dedf84149901